<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "484"
iso_country = "Mexico"
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2025
gdp = 1840e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

'Output directory already exists.'

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Mexico': 

Number of trade records present in dataset: 28015
Number of partner countries present in dataset: 112
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$171672.1m
Total weight within the dataset: 74158.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,484,Mexico,040,Austria,Import,Sea,852351,852351 | Semiconductor media;,8523,"8523 | Discs, tapes, solid-",85,85 | Electrical machinery,4752,"Records, tapes and other recorded media for so...",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,41168,41168,41168,NaN,0.0,0.0,411.68,14003.0,8701.058113,3582.051604,['Container'],['Container'],9.669244,9.640487,9.643443,9.643443,9.657869,9.657869,10.186740,19.059125,11.108261,10.352305,10.394842,10.694714,9.945732,18.576927,11.802167,10.783132,10.492177,11.062664,9.733951,20.648897,14.466866,12.881007,11.838051,11.577614,9.526209,21.529582,17.382764,16.020346,12.934428,12.495440,9.322670,22.694102,21.889331,21.740156,13.711464,13.173484,9.115174,22.036945,21.500254,23.119523,18.066023,13.422611,8.903963,19.144732,20.258100,22.772903,17.060251,13.583595,8.705214,16.959561,17.407880,20.823967,16.558010,13.688859,4752
1,1,484,Mexico,124,Canada,Import,Sea,852351,852351 | Semiconductor media;,8523,"8523 | Discs, tapes, solid-",85,85 | Electrical machinery,4752,"Records, tapes and other recorded media for so...",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,6,6,6,NaN,0.0,0.0,0.06,4509.0,2801.761839,0.168106,['Container'],['Container'],0.000454,0.000452,0.000453,0.000453,0.000453,0.000453,0.000478,0.000894,0.000521,0.000486,0.000488,0.000502,0.000467,0.000872,0.000554,0.000506,0.000492,0.000519,0.000457,0.000969,0.000679,0.000605,0.000556,0.000543,0.000447,0.001010,0.000816,0.000752,0.000607,0.000586,0.000438,0.001065,0.001027,0.001020,0.000643,0.000618,0.000428,0.001034,0.001009,0.001085,0.000848,0.000630,0.000418,0.000898,0.000951,0.001069,0.000801,0.000637,0.000409,0.000796,0.000817,0.000977,0.000777,0.000642,4752


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $9124.3m (5.3%), ... 

                             cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
02 | Meat and edible meat  1853075377  6.933865e+08      1.1      0.9
10 | Cereals               1220746466  4.691004e+09      0.7      6.3
15 | Animal, vegetable or   744583945  6.441071e+08      0.4      0.9
22 | Beverages, spirits a   696271145  1.153549e+09      0.4      1.6
12 | Oil seeds and oleagi   569742557  8.736454e+08      0.3      1.2
08 | Fruit and nuts, edib   497748084  2.773902e+08      0.3      0.4
03 | Fish and crustaceans   477103319  8.744428e+07      0.3      0.1
21 | Miscellaneous edible   421186057  9.284312e+07      0.2      0.1
09 | Coffee, tea, mate an   404675379  1.093536e+08      0.2      0.1
04 | Dairy produce; birds   369090254  7.526889e+07      0.2      0.1
20 | Preparations of vege   364395682  2.097894e+08      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $7714.0m (4.5%), ... 

                    cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       5248408598  6.186862e+09      3.1      8.3
Food & Beverages  2455807869  2.253581e+09      1.4      3.0
Fishing              9810797  1.385566e+06      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $6986.2m (4.1%), ... 

                                 cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     1926984123  9.682226e+08      1.1      1.3
OMT - Meat products n.e.c.     1083915304  5.374744e+08      0.6      0.7
GRO - Cereal grains n.e.c.      846007299  3.499509e+09      0.5      4.7
CMT - Bovine meat prods         778904148  1.583208e+08      0.5      0.2
VOL - Vegetable oils and fats   665007792  7.100346e+08      0.4      1.0
MIL - Dairy products            489778205  9.717218e+07      0.3      0.1
V_F - Vegetables, fruit, nuts   453477333  3.074211e+08      0.3      0.4
OCR - Crops n.e.c.              432988935  1.070549e+08      0.3      0.1
OSD - Oil seeds                 179816375  2.609934e+08      0.1      0.4
PCR - Processed rice             78785214  1.368499e+08      0.0      0.2
OAP - Animal products n.e.c.     36637105  8.6

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 5.3% 
GTAP 4.1% 
EORA 4.5%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16981/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,484,Mexico,040,Austria,Import,Sea,852351,852351 | Semiconductor media;,8523,"8523 | Discs, tapes, solid-",85,85 | Electrical machinery,4752,"Records, tapes and other recorded media for so...",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,41168,41168,41168,NaN,0.0,0.0,411.68,14003.0,8701.058113,3582.051604,['Container'],['Container'],9.669244,9.640487,9.643443,9.643443,9.657869,9.657869,10.186740,19.059125,11.108261,10.352305,10.394842,10.694714,9.945732,18.576927,11.802167,10.783132,10.492177,11.062664,9.733951,20.648897,14.466866,12.881007,11.838051,11.577614,9.526209,21.529582,17.382764,16.020346,12.934428,12.495440,9.322670,22.694102,21.889331,21.740156,13.711464,13.173484,9.115174,22.036945,21.500254,23.119523,18.066023,13.422611,8.903963,19.144732,20.258100,22.772903,17.060251,13.583595,8.705214,16.959561,17.407880,20.823967,16.558010,13.688859,4752,NaN
1,1,484,Mexico,124,Canada,Import,Sea,852351,852351 | Semiconductor media;,8523,"8523 | Discs, tapes, solid-",85,85 | Electrical machinery,4752,"Records, tapes and other recorded media for so...",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,6,6,6,NaN,0.0,0.0,0.06,4509.0,2801.761839,0.168106,['Container'],['Container'],0.000454,0.000452,0.000453,0.000453,0.000453,0.000453,0.000478,0.000894,0.000521,0.000486,0.000488,0.000502,0.000467,0.000872,0.000554,0.000506,0.000492,0.000519,0.000457,0.000969,0.000679,0.000605,0.000556,0.000543,0.000447,0.001010,0.000816,0.000752,0.000607,0.000586,0.000438,0.001065,0.001027,0.001020,0.000643,0.000618,0.000428,0.001034,0.001009,0.001085,0.000848,0.000630,0.000418,0.000898,0.000951,0.001069,0.000801,0.000637,0.000409,0.000796,0.000817,0.000977,0.000777,0.000642,4752,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $4822.4m of $171672.1m (2.8%)
Volume: 6640.2 of 74158.0 million kilograms (9.0%)
Value-to-Weight: 0.73 from 2.31 US$ per kg (+-68.6%)
Average distance to market (volume-weighted): 3142.0 from 4525.7 nm (+-30.6%)

            food_group  N_com    cifvalue    vol_kg_vtw  pct_val  pct_vol
0        Meat and Prep     20  1233225316  5.578609e+08      0.7      0.8
1  Unprocessed Cereals      8  1134022383  4.540364e+09      0.7      6.1
2    Processed Cereals    323   893810888  6.452703e+08      0.5      0.9
3               Fruits     41   471466586  2.652096e+08      0.3      0.4
4                Dairy     26   369090254  7.526889e+07      0.2      0.1
5        Fats and Oils     78   351607301  1.915215e+08      0.2      0.3
6      Sugar and Honey     59   209960172  1.967253e+08      0.1      0.3
7           Vegetables     71   159198608  1.680017e+08      0.1      0.2



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.504002e+08,2.628631e+08,1.619641e+08,1.518409e+08,1.523362e+08,1.569268e+08,1.395331e+08,3.063266e+08,2.958779e+08,2.985993e+08,1.957952e+08,1.885706e+08,1.317067e+08,2.398496e+08,2.457788e+08,2.890226e+08,2.346795e+08,1.955946e+08
1,Fats and Oils,3.184478e+08,5.593820e+08,3.432871e+08,3.212506e+08,3.224243e+08,3.324672e+08,2.949761e+08,6.574559e+08,6.329391e+08,6.391205e+08,4.178996e+08,3.999153e+08,2.782915e+08,5.096067e+08,5.216281e+08,6.141449e+08,5.000989e+08,4.143102e+08
2,Fruits,2.984091e+08,5.501180e+08,3.243493e+08,3.027378e+08,3.039675e+08,3.128525e+08,2.738912e+08,6.547588e+08,6.300374e+08,6.285051e+08,3.992489e+08,3.832246e+08,2.563848e+08,4.928044e+08,5.056320e+08,6.022684e+08,4.812858e+08,3.981052e+08
3,Meat and Prep,1.071641e+09,2.005012e+09,1.168585e+09,1.089058e+09,1.093533e+09,1.125080e+09,9.807411e+08,2.387410e+09,2.302749e+09,2.287056e+09,1.442440e+09,1.385845e+09,9.157850e+08,1.784139e+09,1.831302e+09,2.190673e+09,1.741896e+09,1.440062e+09
4,Processed Cereals,1.338286e+09,2.480883e+09,1.456364e+09,1.358526e+09,1.364067e+09,1.403791e+09,1.226995e+09,2.953650e+09,2.845291e+09,2.833317e+09,1.795453e+09,1.723061e+09,1.147511e+09,2.217038e+09,2.275134e+09,2.714274e+09,2.165087e+09,1.789972e+09
5,Sugar and Honey,3.264709e+08,5.965134e+08,3.542110e+08,3.308443e+08,3.321736e+08,3.419903e+08,3.001668e+08,7.092003e+08,6.820337e+08,6.820317e+08,4.353911e+08,4.174833e+08,2.813977e+08,5.363058e+08,5.501145e+08,6.537294e+08,5.240772e+08,4.335303e+08
6,Unprocessed Cereals,2.211097e+09,3.928452e+09,2.383252e+09,2.232586e+09,2.241155e+09,2.310293e+09,2.043537e+09,4.687403e+09,4.509318e+09,4.530878e+09,2.955856e+09,2.787774e+09,1.924366e+09,3.596532e+09,3.687250e+09,4.338287e+09,3.510457e+09,2.885393e+09
7,Vegetables,2.670407e+08,4.923166e+08,2.902561e+08,2.709075e+08,2.720093e+08,2.799656e+08,2.450985e+08,5.859766e+08,5.638763e+08,5.625114e+08,3.572968e+08,3.429423e+08,2.294304e+08,4.409593e+08,4.524516e+08,5.389632e+08,4.307377e+08,3.562571e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.30,0.03,0.00,0.01,0.02,0.45,0.42,0.43,0.15,0.13,0.29,0.31,0.43,0.28,0.17
1,Fats and Oils,0.69,0.07,0.01,0.01,0.04,1.03,0.96,0.98,0.35,0.30,0.66,0.69,0.96,0.63,0.39
2,Fruits,0.53,0.06,0.01,0.01,0.03,0.81,0.76,0.75,0.27,0.23,0.50,0.53,0.73,0.48,0.30
3,Meat and Prep,0.76,0.08,0.01,0.02,0.04,1.14,1.07,1.06,0.37,0.33,0.70,0.74,1.03,0.67,0.43
4,Processed Cereals,1.28,0.13,0.02,0.03,0.07,1.93,1.81,1.80,0.64,0.56,1.20,1.26,1.75,1.14,0.72
5,Sugar and Honey,1.29,0.13,0.02,0.03,0.07,1.95,1.82,1.82,0.64,0.56,1.21,1.28,1.77,1.16,0.72
6,Unprocessed Cereals,1.51,0.15,0.02,0.03,0.09,2.33,2.17,2.19,0.80,0.66,1.47,1.55,2.13,1.40,0.85
7,Vegetables,1.42,0.15,0.02,0.03,0.08,2.14,2.00,1.99,0.70,0.61,1.33,1.40,1.94,1.26,0.80
